# Fase 2: Processed → Curated
## Deduplicación, columnas derivadas y particionamiento

**Origen:** `abfss://processed@stgabritenreiroprueba.dfs.core.windows.net/`  
**Destino:** `abfss://curated@stgabritenreiroprueba.dfs.core.windows.net/` (particionado por Company)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

try:
    from notebookutils import mssparkutils
# Storage account (hardcoded default, override via pipeline parameters if needed)
STORAGE_ACCOUNT = 'stgabritenreiroprueba'

# Try to override with pipeline parameter
try:
    from notebookutils import mssparkutils
    param = mssparkutils.notebook.getParameter('storage_account')
    if param and param.strip():
        STORAGE_ACCOUNT = param.strip()
except Exception:
    pass

PROCESSED_PATH = f'abfss://processed@{STORAGE_ACCOUNT}.dfs.core.windows.net/'
CURATED_PATH = f'abfss://curated@{STORAGE_ACCOUNT}.dfs.core.windows.net/'

assert PROCESSED_PATH and CURATED_PATH, 'Paths cannot be empty'

print(f'Storage: {STORAGE_ACCOUNT}')
print(f'Processed: {PROCESSED_PATH}')
print(f'Curated: {CURATED_PATH}')
print(f'Spark version: {spark.version}')

---
## 1. Lectura de datos procesados

In [ ]:
df_processed = spark.read.parquet(PROCESSED_PATH)

print(f'Filas cargadas: {df_processed.count():,}')
print(f'Columnas: {len(df_processed.columns)}')
df_processed.printSchema()

In [ ]:
df_processed.show(3, truncate=False, vertical=True)

---
## 2. Eliminación de duplicados exactos

In [ ]:
dup_count = df_processed.count() - df_processed.distinct().count()
print(f'Duplicados exactos detectados: {dup_count}')

df_deduped = df_processed.dropDuplicates()
print(f'Filas tras deduplicar: {df_deduped.count():,}')

---
## 3. Feature Engineering — Columnas derivadas

In [ ]:
df_curated = (df_deduped
    # Métricas de valor
    .withColumn('Price_per_RAM_GB',
        F.round(F.col('Price') / F.col('Ram_GB'), 2))
    .withColumn('Price_per_Storage_GB',
        F.round(F.col('Price') / F.col('Total_Storage_GB'), 4))
    # Extracción de marcas
    .withColumn('CPU_Brand',
        F.when(F.col('Cpu').rlike('(?i)intel'), 'Intel')
         .when(F.col('Cpu').rlike('(?i)amd'), 'AMD')
         .when(F.col('Cpu').rlike('(?i)apple'), 'Apple')
         .otherwise('Other'))
    .withColumn('GPU_Brand',
        F.when(F.col('Gpu').rlike('(?i)nvidia'), 'NVIDIA')
         .when(F.col('Gpu').rlike('(?i)amd'), 'AMD')
         .when(F.col('Gpu').rlike('(?i)intel'), 'Intel')
         .otherwise('Other'))
    # Flag GPU dedicada
    .withColumn('Has_Dedicated_GPU',
        (F.col('Dedicated_Gpu') == 1).cast('int'))
    # Categoría de pantalla
    .withColumn('Screen_Category',
        F.when(F.col('Inches') < 13, 'Compact (<13")')
         .when(F.col('Inches') < 15, 'Standard (13-15")')
         .when(F.col('Inches') < 17, 'Large (15-17")')
         .otherwise('XL (>17")'))
)

print('Schema curated:')
df_curated.printSchema()

In [ ]:
# Verificar nuevas columnas
df_curated.select(
    'Company', 'TypeName', 'Price', 'Ram_GB', 'Total_Storage_GB',
    'Price_per_RAM_GB', 'Price_per_Storage_GB',
    'CPU_Brand', 'GPU_Brand', 'Has_Dedicated_GPU',
    'Inches', 'Screen_Category'
).show(10, truncate=False)

---
## 4. Escritura a Curated (particionado por Company)

In [ ]:
df_curated.write \
    .mode('overwrite') \
    .option('compression', 'snappy') \
    .partitionBy('Company') \
    .parquet(CURATED_PATH)

print('✅ Datos escritos en curated/')

In [ ]:
# Verificación
df_check = spark.read.parquet(CURATED_PATH)
print(f'Curated - Filas: {df_check.count():,} | Columnas: {len(df_check.columns)}')
print(f'\nColumnas: {df_check.columns}')

# Verificar particionamiento
print('\nDistribución por Company:')
df_check.groupBy('Company').count().orderBy(F.desc('count')).show(truncate=False)